# RDT-1B Feature-level BlurIG
**What this does:** Runs ManiSkill simulation to get a real robot camera frame, then applies Blur Integrated Gradients in SigLIP patch-embedding space to show which parts of the image most influenced RDT's predicted gripper action.

**Before running:** Set `Runtime > Change runtime type > A100 GPU`

**Runtime:** ~5-10 min on A100 (first run downloads ~2GB of models)

In [ ]:
# Cell 1 — Clone repos and install dependencies
# Run this cell ONCE. It will automatically restart the runtime at the end.
# After the restart, skip straight to Cell 2.
import os
if os.path.exists('/content/.deps_installed'):
    print('Already installed — skip to Cell 2.')
else:
    os.system('git clone https://github.com/thu-ml/RoboticsDiffusionTransformer')
    os.system('git clone https://github.com/rdgbrandon/rdt-igtesting')

    # Headless OpenGL for ManiSkill (no display server on Colab)
    os.system('apt-get install -qq libegl1-mesa-dev libgles2-mesa-dev libosmesa6-dev')

    # Only install what Colab is missing — skip requirements.txt, it pins
    # old diffusers/protobuf that conflict with Colab's preinstalled packages
    os.system('pip install -q einops timm pyyaml mani-skill')

    # Fix version conflicts
    os.system('pip install -q "numpy<2"')           # cv2 compiled against numpy 1.x
    os.system('pip install -q "protobuf>=5.28.0"')  # transformers SigLIP needs protobuf 5.x

    open('/content/.deps_installed', 'w').close()
    print('Install done. Restarting runtime...')
    os.kill(os.getpid(), 9)   # force restart — run Cell 2 next

In [ ]:
# Cell 2 — Download pre-encoded language embedding (no T5-XXL needed)
# The RDT authors pre-encoded these for ManiSkill tasks — 165 KB vs 40 GB for T5.
from huggingface_hub import hf_hub_download
import shutil

path = hf_hub_download(
    repo_id='robotics-diffusion-transformer/maniskill-model',
    filename='lang_embeds/text_embed_PickCube-v1.pt',
)
shutil.copy(path, 'rdt-igtesting/lang_embed.pt')
print('Language embedding ready.')

In [ ]:
# Cell 3 — Run BlurIG
import os
os.environ['RDT_REPO']       = '/content/RoboticsDiffusionTransformer'
os.environ['LANG_EMBED']     = 'lang_embed.pt'
os.environ['MANISKILL_TASK'] = 'PickCube-v1'
os.chdir('/content/rdt-igtesting')

!git pull
%run rdt_blurig.py

In [ ]:
# Cell 4 — Display result
from IPython.display import Image
Image('rdt_blurig_output.png')

---
## Optional: try other tasks
Available tasks and their pre-encoded lang embeds:
- `PickCube-v1` — pick up the cube
- `StackCube-v1` — stack one cube on another
- `PushCube-v1` — push the cube to a target
- `PegInsertionSide-v1` — insert a peg into a hole
- `PlugCharger-v1` — plug a charger into a socket

In [ ]:
# Optional — swap task and re-run
from huggingface_hub import hf_hub_download
import shutil, os

TASK = 'StackCube-v1'   # change this

path = hf_hub_download(
    repo_id='robotics-diffusion-transformer/maniskill-model',
    filename=f'lang_embeds/text_embed_{TASK}.pt',
)
shutil.copy(path, 'rdt-igtesting/lang_embed.pt')

os.environ['MANISKILL_TASK'] = TASK
os.chdir('/content/rdt-igtesting')
%run rdt_blurig.py

from IPython.display import Image
Image('rdt_blurig_output.png')

---
## Tuning BlurIG parameters
Edit these at the top of `rdt_blurig.py` before running Cell 3:

| Parameter | Default | Effect |
|---|---|---|
| `N_BLURIG_STEPS` | 20 | More steps = smoother heatmap, slower |
| `N_DDPM_STEPS` | 5 | More steps = more faithful to full RDT, slower |
| `SIGMA_MAX` | 2.0 | Higher = blurrier baseline, stronger contrast in heatmap |
| `SCORE_HORIZON` | 8 | How many future action steps to include in the score |